# Notebook 3 - Text Analysis and Word Frequency
Aman Kumar - 251810700231
Alok Chauhan - 251810700318

in this notebook we are finding which words
appear the most in spam vs ham messages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

data = pd.read_csv("outputs/spam_cleaned.csv")
spam = data[data["label"] == "spam"]
ham  = data[data["label"] == "ham"]

print("data loaded!")
print("spam:", len(spam), "| ham:", len(ham))

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/spam_cleaned.csv'

we need to split messages into individual words
and remove common words like "the", "is", "a"
these are called stopwords - they are not useful

In [ ]:
# common words we want to ignore
STOPWORDS = [
    "i","me","my","we","our","you","your","he","him","his",
    "she","her","it","its","they","them","this","that","am",
    "is","are","was","were","be","been","have","has","had",
    "do","does","did","a","an","the","and","but","if","or",
    "of","at","by","for","with","to","from","in","on","not",
    "no","so","u","ur","r","ok","hi","hey","get","go","got",
    "da","lor","n","la","ah","lol","wat","wah","k","will",
    "just","now","like","know","back","time","want","good"
]

def get_words(message):
    # make lowercase
    message = str(message).lower()
    # split into words
    words = message.split()
    # keep only normal words longer than 2 letters
    clean = []
    for word in words:
        # remove punctuation from word
        word = word.strip(".,!?:;()[]\"'")
        # only keep if not a stopword and longer than 2 letters
        if word not in STOPWORDS and len(word) > 2:
            clean.append(word)
    return clean

# test it
test = "FREE entry! Call now to claim your PRIZE!"
print("test message:", test)
print("words found:", get_words(test))

now counting all words in spam and ham separately

In [ ]:
# collecting all words from spam messages
spam_words = []
for msg in spam["message"]:
    spam_words = spam_words + get_words(msg)

# collecting all words from ham messages
ham_words = []
for msg in ham["message"]:
    ham_words = ham_words + get_words(msg)

# counting how many times each word appears
spam_count = Counter(spam_words)
ham_count  = Counter(ham_words)

print("total unique words in spam:", len(spam_count))
print("total unique words in ham:", len(ham_count))

print("\ntop 15 words in spam:")
for word, count in spam_count.most_common(15):
    print(" ", word, "-", count)

print("\ntop 15 words in ham:")
for word, count in ham_count.most_common(15):
    print(" ", word, "-", count)

now making bigrams - these are 2 word combinations
like "please call" or "claim prize"
they tell us more than single words

In [ ]:
def get_bigrams(words):
    # combine every 2 words next to each other
    bigrams = []
    for i in range(len(words) - 1):
        pair = words[i] + " " + words[i+1]
        bigrams.append(pair)
    return bigrams

# getting bigrams for spam
spam_bigrams = []
for msg in spam["message"]:
    words = get_words(msg)
    spam_bigrams = spam_bigrams + get_bigrams(words)

# getting bigrams for ham
ham_bigrams = []
for msg in ham["message"]:
    words = get_words(msg)
    ham_bigrams = ham_bigrams + get_bigrams(words)

spam_bigram_count = Counter(spam_bigrams)
ham_bigram_count  = Counter(ham_bigrams)

print("top 10 spam bigrams:")
for phrase, count in spam_bigram_count.most_common(10):
    print(" ", phrase, "-", count)

print("\ntop 10 ham bigrams:")
for phrase, count in ham_bigram_count.most_common(10):
    print(" ", phrase, "-", count)

checking some important keywords
seeing how much more common they are in spam vs ham
this is called odds ratio

In [ ]:
# list of words to check
keywords = ["free", "call", "txt", "prize",
            "win", "claim", "urgent", "cash",
            "mobile", "stop", "reply", "offer"]

print(f"{'word':<12} {'spam %':>8} {'ham %':>8} {'how many times more in spam':>28}")
print("-" * 60)

for word in keywords:
    # how many spam messages have this word
    sp = spam["message"].str.contains(
        word, case=False, na=False).mean() * 100
    # how many ham messages have this word
    hm = ham["message"].str.contains(
        word, case=False, na=False).mean() * 100
    # odds ratio
    odds = sp / max(hm, 0.01)
    print(f"{word:<12} {sp:>7.1f}%  {hm:>7.1f}%  {odds:>25.1f}x")

making word frequency charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Top Words in Spam vs Ham", fontsize=14)

# top 20 spam words
top_spam = pd.Series(dict(spam_count.most_common(20)))
top_spam = top_spam.sort_values()
axes[0].barh(top_spam.index, top_spam.values, color="#E74C3C")
axes[0].set_title("Top 20 Spam Words", color="#E74C3C")
axes[0].set_xlabel("how many times it appears")

# top 20 ham words
top_ham = pd.Series(dict(ham_count.most_common(20)))
top_ham = top_ham.sort_values()
axes[1].barh(top_ham.index, top_ham.values, color="#2ECC71")
axes[1].set_title("Top 20 Ham Words", color="#2ECC71")
axes[1].set_xlabel("how many times it appears")

plt.tight_layout()
plt.savefig("outputs/03_word_frequency.png", dpi=120)
plt.show()
print("chart saved!")

## what we found

- top spam words are: call, free, txt, mobile, claim, prize, cash
- top ham words are: good, time, love, home, day, sorry
- "please call" is the most common spam bigram
- word FREE is 19 times more common in spam than ham
- word CALL is 11 times more common in spam than ham
- urgent appears only in spam basically

notebook 4 is done by alok - segmentation